# MasterMind Credit Scoring: An Explainable Two-Tier Credit Risk Pipeline

## Project Objective

This notebook is designed as a presentation-first walkthrough of the project rather than a development workspace. The goal is to explain how the system moves from a credit default prediction problem to an explainable, fairness-audited scoring pipeline with a clean live-demo path.

The notebook intentionally favors story, visuals, and compact evidence over long code blocks. Wherever possible, it is structured to reuse persisted artifacts and lightweight repo imports instead of retraining the full pipeline.

## Presentation Flow

We will move through the project in the same order that a presentation audience would naturally expect:

1. Problem statement
2. Dataset overview
3. Feature engineering strategy
4. Modeling approach
5. Final results
6. Improvement journey
7. Explainability
8. Demo prediction path
9. Conclusion

The final truthful baseline for this presentation is the deployed FULL candidate `weighted_blend_full`.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "README.md").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "README.md").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

pd.options.display.max_colwidth = 120


## 1. Problem Statement

Credit scoring is a high-impact prediction problem: the model needs to identify repayment risk accurately, but it also needs to support explanation, calibration, and practical deployment decisions.

In this project, the goal is not only to rank risky applicants well, but to produce a scoring pipeline that is:

- predictive enough to separate lower-risk and higher-risk applicants
- calibrated enough for probability-based decision thresholds
- explainable enough to support business-facing reasoning
- structured enough to support both richer FULL inputs and lighter REDUCED inputs
- auditable enough to include proxy fairness checks before runtime use

That is why the final story is not just "we trained a model." It is "we built a deployable scoring system."

## 2. Dataset Overview

The project is built on the Home Credit Default Risk dataset. This is a realistic credit scoring dataset because it combines a primary application table with multiple linked historical tables that describe prior credit behavior, repayment patterns, and account status over time.

From a presentation perspective, the important idea is simple:

- the application table describes the current applicant
- the child tables add historical behavior and repayment context
- the pipeline uses a repaired processed split regime so evaluation stays truthful

This dataset is challenging in exactly the way real credit risk data is challenging:

- the useful signal is relational rather than fully flat
- several high-signal columns are partially missing
- richer history improves performance, but not every scoring path can assume full coverage

That is why the project supports both FULL and REDUCED tiers. The cells below use lightweight inspection only so the section stays presentation-friendly and fast.

In [ ]:
from configs.config import TEST_FRAC, TRAIN_FRAC, VAL_MODEL_FRAC, VAL_POLICY_FRAC

MAJOR_SOURCE_FILES = [
    {
        "table": "application_train",
        "file": "application_train.csv",
        "grain": "1 row per application",
        "join_key": "SK_ID_CURR",
        "role": "Primary supervised table with the default target",
    },
    {
        "table": "bureau",
        "file": "bureau.csv",
        "grain": "1 row per external credit line",
        "join_key": "SK_ID_CURR / SK_ID_BUREAU",
        "role": "External bureau credit history",
    },
    {
        "table": "bureau_balance",
        "file": "bureau_balance.csv",
        "grain": "monthly bureau status",
        "join_key": "SK_ID_BUREAU",
        "role": "Delinquency status over time for bureau accounts",
    },
    {
        "table": "installments_payments",
        "file": "installments_payments.csv",
        "grain": "1 installment payment event",
        "join_key": "SK_ID_CURR / SK_ID_PREV",
        "role": "Repayment behavior and missed-payment patterns",
    },
    {
        "table": "POS_CASH_balance",
        "file": "POS_CASH_balance.csv",
        "grain": "monthly POS cash snapshot",
        "join_key": "SK_ID_CURR / SK_ID_PREV",
        "role": "Point-of-sale and cash-loan account behavior",
    },
    {
        "table": "previous_application",
        "file": "previous_application.csv",
        "grain": "1 prior application",
        "join_key": "SK_ID_CURR / SK_ID_PREV",
        "role": "History of prior credit applications and outcomes",
    },
]


def quick_csv_profile(file_name: str) -> dict:
    path = RAW_DIR / file_name
    header = pd.read_csv(path, nrows=0)
    return {
        "columns": int(header.shape[1]),
        "size_mb": round(path.stat().st_size / (1024 ** 2), 1),
        "sample_columns": ", ".join(header.columns[:6]),
    }


source_inventory = pd.DataFrame(
    [{**spec, **quick_csv_profile(spec["file"])} for spec in MAJOR_SOURCE_FILES]
)

display(Markdown("### Major raw source files"))
display(source_inventory.style.format({"size_mb": "{:.1f}"}).hide(axis="index"))
display(Markdown("> These linked tables create the relational history used by the FULL tier."))

### Why this dataset is realistic for credit scoring

A good credit model cannot rely only on the current application form. The harder and more realistic version of the problem is to combine current applicant information with fragmented historical evidence, while also handling partial missingness and uneven data coverage.

That is the core data challenge in this project. The FULL tier benefits from richer relational history, while the REDUCED tier keeps the system usable when only the main application record is available.

In [ ]:
APP_OVERVIEW_COLS = [
    "SK_ID_CURR",
    "TARGET",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "DAYS_EMPLOYED",
    "DAYS_BIRTH",
    "NAME_CONTRACT_TYPE",
]

if "_app_train_overview" not in globals():
    _app_train_overview = pd.read_csv(RAW_DIR / "application_train.csv", usecols=APP_OVERVIEW_COLS)

app_train_overview = _app_train_overview.copy()
application_train_columns = pd.read_csv(RAW_DIR / "application_train.csv", nrows=0).columns
application_test_columns = pd.read_csv(RAW_DIR / "application_test.csv", nrows=0).columns
application_test_rows = len(pd.read_csv(RAW_DIR / "application_test.csv", usecols=["SK_ID_CURR"]))

dataset_shapes = pd.DataFrame(
    [
        {
            "table": "application_train",
            "rows": int(len(app_train_overview)),
            "columns": int(len(application_train_columns)),
            "note": "Main supervised application table",
        },
        {
            "table": "application_test",
            "rows": int(application_test_rows),
            "columns": int(len(application_test_columns)),
            "note": "Scoring-only application table",
        },
    ]
)

processed_manifest_path = PROCESSED_DIR / "processed_artifact_manifest.json"
processed_manifest = json.loads(processed_manifest_path.read_text()) if processed_manifest_path.exists() else {}
split_summary = processed_manifest.get("split_summary", {})
split_specs = [
    ("train", TRAIN_FRAC),
    ("val_model", VAL_MODEL_FRAC),
    ("val_policy", VAL_POLICY_FRAC),
    ("test", TEST_FRAC),
]
split_overview = pd.DataFrame(
    [
        {
            "split": split_name,
            "configured_fraction": configured_fraction,
            "rows": split_summary.get(split_name, {}).get("rows"),
            "positive_rate": split_summary.get(split_name, {}).get("target_rate"),
            "available_now": (PROCESSED_DIR / f"{split_name}.pkl").exists(),
        }
        for split_name, configured_fraction in split_specs
    ]
)

split_note = (
    "Processed split counts below are loaded from the saved Module 1 manifest."
    if split_summary
    else "Saved split counts were not found, so the table shows the configured split regime plus artifact availability."
)

representative_columns = pd.DataFrame(
    [
        {"column": "SK_ID_CURR", "role": "Primary key", "why_it_matters": "Links the application to historical child tables"},
        {"column": "TARGET", "role": "Training label", "why_it_matters": "Indicates default vs non-default for supervised learning"},
        {"column": "AMT_INCOME_TOTAL", "role": "Applicant income", "why_it_matters": "Core affordability and repayment-capacity signal"},
        {"column": "AMT_CREDIT", "role": "Requested credit", "why_it_matters": "Drives credit burden and exposure"},
        {"column": "AMT_ANNUITY", "role": "Scheduled payment", "why_it_matters": "Helps express repayment pressure"},
        {"column": "EXT_SOURCE_1/2/3", "role": "External risk sources", "why_it_matters": "High-signal features with uneven coverage and missingness"},
        {"column": "DAYS_EMPLOYED", "role": "Employment history proxy", "why_it_matters": "Captures stability and applicant profile context"},
        {"column": "NAME_CONTRACT_TYPE", "role": "Application category", "why_it_matters": "Separates product context such as cash vs revolving loans"},
    ]
)

display(Markdown("### Dataset shapes"))
display(dataset_shapes.style.hide(axis="index"))
display(Markdown("### Processed split overview"))
display(Markdown(split_note))
display(split_overview.style.format({"configured_fraction": "{:.0%}", "positive_rate": "{:.1%}"}).hide(axis="index"))
display(Markdown("### Representative columns"))
display(representative_columns.style.hide(axis="index"))

preview_cols = [
    "TARGET",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "EXT_SOURCE_2",
    "DAYS_EMPLOYED",
    "NAME_CONTRACT_TYPE",
]
display(Markdown("### Example rows from the main application table"))
app_train_overview[preview_cols].head(5).style.format(
    {
        "AMT_INCOME_TOTAL": "{:.0f}",
        "AMT_CREDIT": "{:.0f}",
        "AMT_ANNUITY": "{:.0f}",
        "EXT_SOURCE_2": "{:.3f}",
        "DAYS_EMPLOYED": "{:.0f}",
    }
).hide(axis="index")

### Quick EDA

For presentation purposes, two fast diagnostics carry most of the story:

- the target balance in the supervised application table
- the missingness pattern in a few representative high-signal columns, especially the external source features

This keeps the notebook lightweight while still showing why the data is difficult and realistic.

In [ ]:
import matplotlib.pyplot as plt

target_summary = (
    app_train_overview["TARGET"]
    .value_counts(dropna=False)
    .rename_axis("target")
    .reset_index(name="rows")
    .sort_values("target")
)
target_summary["share"] = target_summary["rows"] / target_summary["rows"].sum()

missingness_summary = (
    app_train_overview[
        [
            "EXT_SOURCE_1",
            "EXT_SOURCE_2",
            "EXT_SOURCE_3",
            "AMT_ANNUITY",
            "DAYS_EMPLOYED",
        ]
    ]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "column"})
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(target_summary["target"].astype(str), target_summary["rows"], color=["#4C78A8", "#E45756"])
axes[0].set_title("Target Distribution")
axes[0].set_xlabel("TARGET")
axes[0].set_ylabel("Rows")
for idx, row in target_summary.iterrows():
    axes[0].text(idx, row["rows"] + 3000, f"{row['share']:.1%}", ha="center", fontsize=10)

axes[1].barh(missingness_summary["column"], missingness_summary["missing_pct"], color="#72B7B2")
axes[1].set_title("Missingness in Representative Columns")
axes[1].set_xlabel("Missing values (%)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

display(Markdown("### Target distribution summary"))
display(target_summary.style.format({"share": "{:.1%}"}).hide(axis="index"))
display(Markdown("### Missingness summary"))
missingness_summary.style.format({"missing_pct": "{:.1f}%"}).hide(axis="index")

## 3. Feature Engineering

The feature engineering story is one of the main reasons the final FULL baseline is credible. The goal was not to add as many columns as possible. The goal was to add the right blocks of signal and keep only the improvements that strengthened the truthful baseline.

At a high level, the pipeline combines three layers of information:

- core application features from the main applicant record
- engineered application features such as ratios, burden signals, and EXT_SOURCE interactions
- relational credit-history aggregates built from linked historical tables

Two accepted improvements matter most in the final story:

- stronger relational aggregate blocks across bureau, previous applications, installments, POS cash, and credit card history
- an expanded EXT_SOURCE interaction block that made the application-level signal more robust

### Why relational credit-history features matter

The current application tells us what the customer is asking for today. The linked history tables tell us how similar obligations were managed over time. That difference matters in credit scoring. Bureau and bureau balance features add external debt and delinquency context, while installments features add direct repayment-behavior signals such as missed payments and days past due.

### Why the EXT_SOURCE block mattered

The external source features are strong signals, but they are also incomplete and unevenly missing. Instead of relying only on each raw column independently, the feature engineering layer adds summary statistics and interaction terms so the model can capture agreement, disagreement, and combined strength across the external risk signals.

### FULL vs REDUCED

- `REDUCED` keeps the application-level view: raw application fields plus engineered application features.
- `FULL` starts with that same foundation and adds relational credit-history aggregates from linked tables.

That is why the FULL tier wins: it sees both present-day affordability and historical repayment behavior.

A final discipline point matters for the presentation: some additional feature variants were tested offline, but they were not carried forward if they did not improve the truthful baseline.

In [ ]:
from src.feature_engineering import (
    ALL_AGGREGATE_FEATURE_COLS,
    BUREAU_AGG_COLS,
    CREDIT_CARD_AGG_COLS,
    DEFAULT_FULL_FEATURE_VIEW,
    ENGINEERED_APP_FEATURE_COLS,
    INSTALLMENTS_AGG_COLS,
    POS_CASH_AGG_COLS,
    PREVIOUS_AGG_COLS,
)

ext_block_features = [name for name in ENGINEERED_APP_FEATURE_COLS if name.startswith("EXT_")]
non_ext_application_features = [name for name in ENGINEERED_APP_FEATURE_COLS if not name.startswith("EXT_")]

tier_summary = pd.DataFrame(
    [
        {
            "tier": "REDUCED",
            "core_concept": "Application-only scoring path with engineered application features",
            "historical_tables_used": "No",
            "what_it_sees": "Current applicant profile, affordability, and EXT_SOURCE interactions",
        },
        {
            "tier": "FULL",
            "core_concept": "REDUCED foundation plus relational credit-history aggregates",
            "historical_tables_used": f"Yes ({len(ALL_AGGREGATE_FEATURE_COLS)} aggregate features)",
            "what_it_sees": "Application profile plus bureau, repayment, prior-loan, POS, and credit-card behavior",
        },
    ]
)

feature_group_summary = pd.DataFrame(
    [
        {
            "feature_group": "Application ratios and burden signals",
            "included_in": "FULL + REDUCED",
            "count": len(non_ext_application_features),
            "examples": "AGE_YEARS, CREDIT_INCOME_RATIO, ANNUITY_INCOME_RATIO",
            "why_it_matters": "Turns raw amounts and dates into more decision-relevant affordability and stability signals",
        },
        {
            "feature_group": "EXT_SOURCE interaction block",
            "included_in": "FULL + REDUCED",
            "count": len(ext_block_features),
            "examples": "EXT_SOURCE_MEAN, EXT_12_PRODUCT, EXT_123_PRODUCT",
            "why_it_matters": "Captures combined strength and disagreement across external risk signals instead of treating each source independently",
        },
        {
            "feature_group": "Bureau + bureau_balance aggregates",
            "included_in": "FULL only",
            "count": len(BUREAU_AGG_COLS),
            "examples": "BUREAU_DEBT_TO_CREDIT_RATIO, BB_MAX_STATUS_MAX, BB_ADVERSE_ACCOUNT_RATE",
            "why_it_matters": "Adds external debt exposure, delinquency severity, and adverse-account history",
        },
        {
            "feature_group": "Installments aggregates",
            "included_in": "FULL only",
            "count": len(INSTALLMENTS_AGG_COLS),
            "examples": "INST_MISSED_RATE, INST_DPD_MEAN, INST_RECENT_365_DPD_MAX",
            "why_it_matters": "Adds direct repayment-behavior evidence such as lateness, underpayment, and recent stress",
        },
        {
            "feature_group": "Other relational history blocks",
            "included_in": "FULL only",
            "count": len(PREVIOUS_AGG_COLS) + len(POS_CASH_AGG_COLS) + len(CREDIT_CARD_AGG_COLS),
            "examples": "PREV_APPROVAL_RATE, POS_DPD_MEAN, CC_UTILIZATION_MEAN",
            "why_it_matters": "Completes the historical view with prior application outcomes, POS behavior, and revolving credit usage",
        },
    ]
)

display(Markdown(f"**Default FULL feature view:** `{DEFAULT_FULL_FEATURE_VIEW}`"))
display(Markdown("### FULL vs REDUCED concept"))
display(tier_summary.style.hide(axis="index"))
display(Markdown("### Major feature groups"))
display(feature_group_summary.style.hide(axis="index"))
display(Markdown("> Note: Additional feature variants were evaluated offline, but only the stronger aggregate blocks and expanded EXT_SOURCE interactions were kept because they supported the truthful baseline story."))

## 4. Modeling Approach

The modeling design is intentionally practical: the project supports two coverage tiers, then uses the richer tier as the final champion when more historical information is available.

### REDUCED tier

The REDUCED path uses application-level inputs plus engineered application features. It is the lighter scoring route and acts as the fallback tier when full relational history is unavailable.

### FULL tier

The FULL path starts from the same application foundation, then adds the accepted historical aggregate stack. In the current truthful baseline, the final FULL runtime candidate is `weighted_blend_full`.

That FULL stack combines:

- an XGBoost model trained on the FULL feature set
- a LightGBM model trained on the same FULL feature set
- a weighted blend that combines both model probabilities

### What the weighted blend means

In simple terms, the blend lets two strong tree models look at the same applicant and then combines their probability estimates using validation-selected weights. Instead of trusting only one model family, the final score keeps the strengths of both. That is the key modeling improvement that became the deployed FULL champion.

### Calibration and decisions

The project does not stop at raw model scores. Probabilities are calibrated and then mapped into a fixed three-band decision policy:

- `APPROVE` for low predicted default risk
- `REVIEW` for intermediate cases
- `DECLINE` for higher predicted default risk

That makes the modeling story more realistic for a production-style presentation: a strong feature stack, a truthful champion model, calibrated probabilities, and a clear operational decision policy.

In [ ]:
from configs.config import APPROVE_THRESHOLD, DECLINE_THRESHOLD, FAIRNESS_AUDIT_VERSION, MODEL_VERSIONS

tier_modeling_summary = pd.DataFrame(
    [
        {
            "tier": "REDUCED",
            "feature_scope": "Application + engineered application features",
            "model_stack": "XGBoost",
            "historical_aggregates": "No",
            "presentation_role": "Fallback tier when only application-level data is available",
        },
        {
            "tier": "FULL",
            "feature_scope": "REDUCED foundation + accepted historical aggregate stack",
            "model_stack": "XGBoost + LightGBM + weighted blend",
            "historical_aggregates": "Yes",
            "presentation_role": "Champion tier and truthful final baseline",
        },
    ]
)

full_stack_summary = pd.DataFrame(
    [
        {"stage": "1. Feature view", "detail": "FULL application features plus accepted aggregate history blocks"},
        {"stage": "2. Base learner A", "detail": "XGBoost trained on FULL features"},
        {"stage": "3. Base learner B", "detail": "LightGBM trained on the same FULL features"},
        {"stage": "4. Weighted blend", "detail": "Validation-selected weighted average of XGBoost and LightGBM probabilities"},
        {"stage": "5. Calibration", "detail": "Probability calibration before policy decisions"},
        {"stage": "6. Decision policy", "detail": f"APPROVE < {APPROVE_THRESHOLD:.2f}, REVIEW < {DECLINE_THRESHOLD:.2f}, else DECLINE"},
    ]
)

runtime_summary = pd.DataFrame(
    [
        {"component": "FULL final baseline", "value": "weighted_blend_full"},
        {"component": "FULL deployed version", "value": "full_weighted_blend_v2.2.0"},
        {"component": "REDUCED deployed version", "value": MODEL_VERSIONS["reduced"]},
        {"component": "Fairness audit version", "value": FAIRNESS_AUDIT_VERSION},
    ]
)

display(Markdown("**Champion model:** `weighted_blend_full` is the truthful FULL baseline used in this presentation."))
display(Markdown("### REDUCED vs FULL"))
display(tier_modeling_summary.style.hide(axis="index"))
display(Markdown("### Final FULL stack"))
display(full_stack_summary.style.hide(axis="index"))
display(Markdown("### Runtime summary"))
runtime_summary.style.hide(axis="index")

## 5. Results

For the presentation, the most important result is that the richer FULL stack clearly outperformed the lighter REDUCED tier and became the truthful champion.

The FULL model wins because it combines three things that the REDUCED tier does not have all at once:

- accepted relational history features
- the expanded EXT_SOURCE interaction block
- the final weighted blend over XGBoost and LightGBM

The REDUCED model remains useful and honest as a fallback, but it does not match the FULL tier's ranking performance.

The notebook should emphasize two kinds of quality:

- **ROC-AUC** for ranking quality
- **Brier score** for probability quality

That combination supports a more honest presentation than showing AUC alone. It makes clear not only which model ranks better, but also which model produces better-behaved probabilities.

The progression table below is also important. It separates what actually helped from what was tested but not retained, so the audience can see the project as disciplined iteration rather than feature accumulation.

In [ ]:
import matplotlib.pyplot as plt

FULL_TEST_ROC_AUC = 0.7740929800973959
FULL_TEST_BRIER = 0.08003396077030062
REDUCED_TEST_ROC_AUC = 0.7511183627949813
ROC_AUC_LIFT = FULL_TEST_ROC_AUC - REDUCED_TEST_ROC_AUC

final_results = pd.DataFrame(
    [
        {
            "tier": "FULL",
            "baseline": "weighted_blend_full",
            "model_stack": "XGBoost + LightGBM + weighted blend",
            "test_roc_auc": FULL_TEST_ROC_AUC,
            "brier_score": FULL_TEST_BRIER,
            "takeaway": "Champion tier",
        },
        {
            "tier": "REDUCED",
            "baseline": "reduced_xgboost",
            "model_stack": "XGBoost",
            "test_roc_auc": REDUCED_TEST_ROC_AUC,
            "brier_score": pd.NA,
            "takeaway": "Fallback tier",
        },
    ]
)

improvement_progression = pd.DataFrame(
    [
        {
            "category": "Accepted improvement",
            "change": "Historical accepted aggregate stack",
            "kept_in_final": "Yes",
            "what_to_say": "Relational credit-history features were strong enough to stay in the final FULL baseline.",
        },
        {
            "category": "Accepted improvement",
            "change": "EXT_SOURCE interaction expansion",
            "kept_in_final": "Yes",
            "what_to_say": "The expanded EXT_SOURCE block improved the application-level signal and remained part of the champion story.",
        },
        {
            "category": "Accepted improvement",
            "change": "Weighted blend improvement",
            "kept_in_final": "Yes",
            "what_to_say": "Combining XGBoost and LightGBM probabilities became the final FULL runtime baseline: `weighted_blend_full`.",
        },
        {
            "category": "Rejected / neutral",
            "change": "Narrow tuning",
            "kept_in_final": "No",
            "what_to_say": "Tested, but not retained because it did not change the truthful baseline story enough.",
        },
        {
            "category": "Rejected / neutral",
            "change": "Regularized meta-blend",
            "kept_in_final": "No",
            "what_to_say": "Evaluated offline, but the runtime baseline remained the weighted blend rather than a regularized meta-learner.",
        },
        {
            "category": "Rejected / neutral",
            "change": "POS_CASH expansion",
            "kept_in_final": "No",
            "what_to_say": "Additional expansion beyond the accepted stack was not retained in the final truthful baseline.",
        },
        {
            "category": "Rejected / neutral",
            "change": "CatBoost",
            "kept_in_final": "No",
            "what_to_say": "Not part of the retained production story because it did not replace the truthful baseline.",
        },
        {
            "category": "Rejected / neutral",
            "change": "CV blend-weighting",
            "kept_in_final": "No",
            "what_to_say": "Cross-validated weighting ideas were not kept because the final truthful runtime stayed with the selected weighted blend path.",
        },
        {
            "category": "Rejected / neutral",
            "change": "Regularization-only experiment",
            "kept_in_final": "No",
            "what_to_say": "Regularization alone was not enough to justify replacing the champion baseline.",
        },
        {
            "category": "Rejected / neutral",
            "change": "Feature pruning",
            "kept_in_final": "No",
            "what_to_say": "Simplification experiments were not retained when they did not improve the truthful baseline.",
        },
    ]
)

display(Markdown(f"**FULL vs REDUCED ROC-AUC lift:** `{ROC_AUC_LIFT:.4f}`"))
display(Markdown("### Final metric comparison"))
display(final_results.style.format({"test_roc_auc": "{:.4f}", "brier_score": "{:.4f}"}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(final_results["tier"], final_results["test_roc_auc"], color=["#4C78A8", "#9C755F"])
ax.set_ylim(0.70, 0.79)
ax.set_ylabel("Test ROC-AUC")
ax.set_title("Champion Performance by Coverage Tier")
for idx, value in enumerate(final_results["test_roc_auc"]):
    ax.text(idx, value + 0.001, f"{value:.4f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

display(Markdown("> The next section separates the accepted improvements from the experiments that were not retained."))

## 6. Comparison of Improvements

A strong presentation does not claim that every experiment helped. It shows disciplined iteration.

In this project, the accepted improvements were the ones that actually strengthened the truthful baseline:

- stronger aggregate feature blocks
- expanded EXT_SOURCE interaction features

Many later experiments were intentionally rejected because they did not beat the baseline strongly enough to justify changing the live story. That decision matters: it keeps the notebook honest and the presentation focused.

In [ ]:
accepted_view = improvement_progression.loc[
    improvement_progression["category"] == "Accepted improvement",
    ["change", "what_to_say"],
].rename(columns={"change": "accepted improvement", "what_to_say": "why it stayed"})

rejected_view = improvement_progression.loc[
    improvement_progression["category"] == "Rejected / neutral",
    ["change", "what_to_say"],
].rename(columns={"change": "tested but not retained", "what_to_say": "why it did not change the baseline"})

display(Markdown("### Kept in the final baseline"))
display(accepted_view.style.hide(axis="index"))
display(Markdown("### Tested but not retained"))
display(rejected_view.style.hide(axis="index"))

## 7. Explainability

Explainability is part of the final system design, not an afterthought.

For presentation purposes, the goal is simple: show that the final FULL baseline is not a black box and that the dominant risk signals make business sense.

This project uses a fast global importance view plus a local SHAP example for the final FULL baseline:

- a **global driver view** that shows which features matter most overall
- a **local SHAP explanation** that shows which features pushed risk up or down for one applicant
- a **business-language interpretation layer** that maps technical features into understandable credit-risk reasoning

Judges do not need the mathematics of SHAP here. The important takeaway is that the model can point to concrete signals such as debt burden, repayment stress, external credit-history risk, and utilization patterns, and those signals are sensible for default prediction.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import warnings
from IPython.display import Markdown, display

from src.explainability import render_reason, top_5_explanations_from_shap

interpretation_guide = pd.DataFrame(
    [
        {"feature_family": "CREDIT_INCOME_RATIO / ANNUITY_INCOME_RATIO", "why_it_matters": "Higher requested debt and repayment burden can signal elevated default risk."},
        {"feature_family": "BUREAU_* / BB_*", "why_it_matters": "External credit history and delinquency records reveal debt stress beyond the current application."},
        {"feature_family": "INST_*", "why_it_matters": "Missed installments and days past due directly capture repayment behavior."},
        {"feature_family": "CC_* / POS_*", "why_it_matters": "Utilization, card behavior, and POS delinquency can reflect ongoing liquidity stress."},
        {"feature_family": "EXT_SOURCE_*", "why_it_matters": "External risk signals are important because they summarize risk information not captured by one application field alone."},
    ]
)


def _normalize_shap_array(raw_shap):
    try:
        import shap  # type: ignore
    except Exception:
        shap = None

    if shap is not None and isinstance(raw_shap, shap.Explanation):
        raw_shap = raw_shap.values
    if isinstance(raw_shap, list):
        raw_shap = raw_shap[1] if len(raw_shap) > 1 else raw_shap[0]

    values = np.asarray(raw_shap, dtype=float)
    if values.ndim == 3:
        class_index = 1 if values.shape[-1] > 1 else 0
        values = values[..., class_index]
    if values.ndim == 1:
        values = values.reshape(1, -1)
    return values


display(Markdown("### How to read the risk signals"))
display(interpretation_guide.style.hide(axis="index"))

artifacts = None
global_driver_table = None

try:
    import src.api.app as api_app
    from src.models.train import load_artifacts

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=FutureWarning)
        warnings.filterwarnings("ignore", message="Trying to unpickle estimator.*")
        warnings.filterwarnings("ignore", message=".*force_all_finite.*")
        artifacts = load_artifacts(
            artifact_dir=str(ARTIFACT_DIR),
            processed_dir=str(PROCESSED_DIR),
            strict_artifacts=False,
        )
        sample_payload = api_app._build_demo_seed_payload()
        sample_input = api_app.build_input_df(sample_payload, "FULL")
        sample_features = artifacts["full_builder"].transform(sample_input)

    xgb_importance = np.asarray(artifacts["full_model"].xgboost_model.feature_importances_, dtype=float)
    lgb_importance = np.asarray(artifacts["full_model"].lightgbm_model.feature_importances_, dtype=float)
    if xgb_importance.sum() > 0:
        xgb_importance = xgb_importance / xgb_importance.sum()
    if lgb_importance.sum() > 0:
        lgb_importance = lgb_importance / lgb_importance.sum()
    weighted_importance = (
        artifacts["full_model"].weight_xgboost * xgb_importance
        + artifacts["full_model"].weight_lightgbm * lgb_importance
    )

    global_driver_table = pd.DataFrame(
        {
            "feature": sample_features.columns,
            "weighted_importance": weighted_importance,
        }
    ).sort_values("weighted_importance", ascending=False).head(8)
    global_driver_table["business_reason"] = [render_reason(feature) for feature in global_driver_table["feature"]]

    display(Markdown("### Global driver view"))
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(global_driver_table["feature"][::-1], global_driver_table["weighted_importance"][::-1], color="#4C78A8")
    ax.set_xlabel("Weighted feature importance")
    ax.set_title("Top global drivers for the FULL baseline")
    plt.tight_layout()
    plt.show()

    display(Markdown("### Top global drivers"))
    display(global_driver_table.style.format({"weighted_importance": "{:.4f}"}).hide(axis="index"))

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=FutureWarning)
        warnings.filterwarnings("ignore", message=".*force_all_finite.*")
        raw_pd = float(np.asarray(artifacts["full_model"].predict_proba(sample_features), dtype=float)[0, 1])
        calibrated_pd = float(np.asarray(artifacts["full_calibrator"].predict([raw_pd]), dtype=float).reshape(-1)[0])
        local_shap = _normalize_shap_array(
            artifacts["full_shap_explainer"](sample_features.to_numpy(dtype=float, copy=False))
        )[0]
        local_shap_series = pd.Series(local_shap, index=sample_features.columns, dtype=float)

    local_top = local_shap_series.abs().sort_values(ascending=False).head(5).index.tolist()
    local_top_series = local_shap_series[local_top]
    local_explanation_table = pd.DataFrame(
        {
            "feature": local_top_series.index,
            "shap_value": local_top_series.values,
            "direction": ["pushes risk up" if value > 0 else "pushes risk down" for value in local_top_series.values],
            "business_reason": [render_reason(feature) for feature in local_top_series.index],
        }
    )

    display(Markdown(f"### Local SHAP example for one FULL applicant (PD = `{calibrated_pd:.3f}`)"))
    fig, ax = plt.subplots(figsize=(8, 4.5))
    bar_colors = ["#E45756" if value > 0 else "#54A24B" for value in local_top_series.values]
    ax.barh(local_explanation_table["feature"][::-1], local_explanation_table["shap_value"][::-1], color=bar_colors[::-1])
    ax.axvline(0.0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xlabel("SHAP contribution")
    ax.set_title("Top local drivers for the sample applicant")
    plt.tight_layout()
    plt.show()

    display(local_explanation_table.style.format({"shap_value": "{:.4f}"}).hide(axis="index"))
    display(Markdown("### Business-language top 5 reasons"))
    display(pd.DataFrame(top_5_explanations_from_shap(local_shap_series)).style.hide(axis="index"))
except Exception as exc:
    display(Markdown(f"> Explainability outputs could not be rendered from the saved FULL artifacts: `{type(exc).__name__}`."))

if global_driver_table is not None:
    top_global_features = ", ".join(global_driver_table["feature"].head(3).tolist())
    display(Markdown(
        f"**Interpretation:** The strongest global drivers in the final FULL stack include `{top_global_features}`. The local SHAP view then shows how those broader signals combine for one applicant."
    ))


## 8. Demo Prediction

A presentation notebook should end with one clean scoring example. The purpose of this section is not to rebuild the app in Jupyter. It is to show that the final model can score one realistic applicant in a practical, readable way.

The demo flow here is intentionally simple:

1. prepare one sample FULL applicant
2. score that applicant with the truthful FULL baseline artifacts
3. show probability of default, decision band, and a short explanation

This keeps the notebook fast and makes the model feel operational without turning the notebook into a full application.

In [ ]:
from IPython.display import Markdown, display
import warnings

import src.api.app as api_app

demo_payload = api_app._build_demo_seed_payload()

demo_input_summary = pd.DataFrame(
    [
        {"input": "Income", "value": f"{demo_payload['application']['AMT_INCOME_TOTAL_CAPPED']:,.0f}"},
        {"input": "Credit requested", "value": f"{demo_payload['application']['AMT_CREDIT']:,.0f}"},
        {"input": "Annuity", "value": f"{demo_payload['application']['AMT_ANNUITY']:,.0f}"},
        {"input": "EXT_SOURCE_1", "value": f"{demo_payload['application']['EXT_SOURCE_1']:.3f}"},
        {"input": "EXT_SOURCE_2", "value": f"{demo_payload['application']['EXT_SOURCE_2']:.3f}"},
        {"input": "EXT_SOURCE_3", "value": f"{demo_payload['application']['EXT_SOURCE_3']:.3f}"},
        {"input": "Bureau loan count", "value": f"{demo_payload['bureau_agg']['BUREAU_LOAN_COUNT']:.0f}"},
        {"input": "Bureau debt ratio", "value": f"{demo_payload['bureau_agg']['BUREAU_DEBT_TO_CREDIT_RATIO']:.2f}"},
        {"input": "Installment DPD mean", "value": f"{demo_payload['installments_agg']['INST_DPD_MEAN']:.1f} days"},
        {"input": "Credit-card utilization", "value": f"{demo_payload['credit_card_agg']['CC_UTILIZATION_MEAN']:.2f}"},
    ]
)

runtime_paths = [
    "full_model.joblib",
    "full_calibrator.joblib",
    "full_shap_explainer.joblib",
    "full_feature_builder.joblib",
]

display(Markdown("### Sample applicant summary"))
display(demo_input_summary.style.hide(axis="index"))

runtime_ready = all((ARTIFACT_DIR / name).exists() for name in runtime_paths)

if runtime_ready:
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=FutureWarning)
            warnings.filterwarnings("ignore", message="Trying to unpickle estimator.*")
            warnings.filterwarnings("ignore", message=".*force_all_finite.*")
            from src.models.train import load_artifacts

            artifacts = load_artifacts(
                artifact_dir=str(ARTIFACT_DIR),
                processed_dir=str(PROCESSED_DIR),
                strict_artifacts=True,
            )
            demo_input_df = api_app.build_input_df(demo_payload, "FULL")
            demo_features = artifacts["full_builder"].transform(demo_input_df)
            raw_pd = api_app._predict_raw_pd(artifacts["full_model"], demo_features)
            calibrated_pd = api_app._calibrate_pd(artifacts["full_calibrator"], raw_pd)
            decision = api_app._decision_from_pd(calibrated_pd)
            explanations = api_app._compute_real_top_5_explanations(artifacts["full_shap_explainer"], demo_features)
        top_features = ", ".join(item["feature"] for item in explanations[:3])

        top_explanation_table = pd.DataFrame(explanations).head(3)

        demo_scorecard = pd.DataFrame(
            [
                {"field": "Coverage tier", "value": "FULL"},
                {"field": "Final baseline", "value": "weighted_blend_full"},
                {"field": "Probability of default", "value": f"{calibrated_pd:.2%}"},
                {"field": "Decision", "value": decision},
                {"field": "Calibrated", "value": "Yes"},
            ]
        )

        display(Markdown("### Demo scoring result"))
        display(demo_scorecard.style.hide(axis="index"))
        display(Markdown("### Top explanation cues"))
        display(top_explanation_table.style.hide(axis="index"))

        if decision == "APPROVE":
            interpretation = (
                f"This applicant looks relatively safer because the calibrated probability of default is below the approve threshold of {api_app.APPROVE_THRESHOLD:.2f}. "
                f"The most influential local signals in this example are {top_features}."
            )
        elif decision == "REVIEW":
            interpretation = (
                f"This applicant sits in the review band, so the model is signaling meaningful risk but not enough for an automatic decline. "
                f"The key local drivers in this case are {top_features}."
            )
        else:
            interpretation = (
                f"This applicant looks materially riskier because the calibrated probability of default lands in the decline band. "
                f"The strongest local risk signals in this case are {top_features}."
            )

        display(Markdown(f"**Interpretation:** {interpretation}"))
    except Exception as exc:
        display(Markdown(f"> Runtime artifacts were found, but notebook scoring failed: `{type(exc).__name__}`."))
else:
    display(Markdown(
        "> Truthful demo scoring is wired, but it cannot run in this clone yet because the FULL runtime artifacts are not present in `artifacts/`. The sample applicant summary is ready, and the scoring call will activate automatically once those files exist."
    ))


## 9. Conclusion

This project presents a practical credit-risk workflow rather than a single isolated model. The final FULL champion, `weighted_blend_full`, won because it combined stronger relational history features, a richer EXT_SOURCE block, and a weighted XGBoost + LightGBM stack.

### Key takeaways

- richer historical aggregates mattered more than late-stage tuning
- the FULL tier outperformed the REDUCED fallback while keeping the system usable when history is incomplete
- calibration, explainability, and a clean demo path make the model easier to trust and present

### Real-world relevance

This mirrors a realistic lending setting: some applicants have rich linked history and some do not, so a two-tier scoring design with calibrated probabilities is more practical than a single all-or-nothing model.

### Limitation

The best performance depends on linked historical coverage, so production quality still depends on the completeness and stability of upstream credit-history data.